In [ ]:
!pip install --upgrade ipython jupyter
%load_ext autoreload
%autoreload 2

In [ ]:
def refresh_repo():
    %cd /kaggle/working
    %rm -rf from-neurons-to-directions
    !git clone https://github.com/jefri021/from-neurons-to-directions.git
    %cd /kaggle/working/from-neurons-to-directions/
    !git pull origin main

refresh_repo()

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/from-neurons-to-directions/src")

import torch
import matplotlib.pyplot as plt

from model_utils import (
    load_model_and_tokenizer,
    apply_chat_template,
    get_num_layers,
    generate,
)
from activation_store import ActivationStore
from refusal_direction import (
    compute_refusal_direction,
    select_best_direction,
    generate_with_ablation,
    generate_with_addition,
)
from metrics import (
    refusal_rate,
    refusal_rate_delta,
    layer_sensitivity,
)
from viz import (
    plot_direction_norms,
    plot_refusal_rates,
    plot_layer_sensitivity,
)

print("Imports OK")

In [ ]:
model, tokenizer = load_model_and_tokenizer("qwen_instruct")

In [ ]:
best_r = torch.load("/kaggle/working/from-neurons-to-directions/data/best_direction.pt")
best_layer = 12
best_pos = -6

In [ ]:
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF-READ")

harmless_ds = load_dataset("tatsu-lab/alpaca", token=hf_token)

harmless = harmless_ds["train"]

# Keep only Alpaca samples with empty input field
harmless_prompts_raw = [
    instruction
    for instruction, inp in zip(harmless["instruction"], harmless["input"])
    if inp.strip() == ""
]

print(f"Harmless prompts : {len(harmless_prompts_raw)}")

harmless_prompts = [apply_chat_template(p, tokenizer) for p in harmless_prompts_raw]

In [ ]:
print("Testing activation addition on harmless prompts...")

addition_responses = generate_with_addition(
    model=model,
    tokenizer=tokenizer,
    prompts=harmless_prompts[:100],
    direction=best_r,
    layer_idx=best_layer,
    alpha=20.0,          # injection strength — tune if output is incoherent
    max_new_tokens=100,
)

addition_refusal_rate = refusal_rate(addition_responses)

print(f"\nRefusal rate after addition (on harmless prompts): {addition_refusal_rate:.0%}")
print("\nSample addition responses:")
for prompt_raw, response in zip(harmless_prompts_raw[:100], addition_responses[:100]):
    print(f"\n  Prompt  : {prompt_raw}")
    print(f"  Response: {response}")